# MedAESQA Data Prep — FaithfulMed (Google 2D)

Owner: Isha and Isabella | September milestone, Task #3

**What this notebook produces.** Four clean CSV files plus a freeze manifest, so nobody
else on the team ever has to open the raw JSON or re-derive our cleaning rules.

| File | One row per | Used by |
|---|---|---|
| `medaesqa_questions.csv` | question (40) | everyone |
| `medaesqa_nuggets.csv` | expert fact (498) | Extractor |
| `medaesqa_sentences.csv` | answer sentence (5,162) | Verifier, Readability |
| `medaesqa_citations.csv` | sentence-PMID pair (~8,124) | Verifier |
| `medaesqa_freeze.txt` | — | reproducibility |

**What this notebook does NOT do.** No imputation, no outlier removal, no rescaling of
the originals. Everything in MedAESQA is text and human labels. There are missing values,
but a missing label means a human did not grade that item, so filling it in would be
inventing data. The real prep work is flattening nested JSON, standardizing inconsistent
category strings, and fixing one genuine ambiguity in how citations are counted.

Missing values, outliers and normalization are still audited explicitly (sections 7-9):
gaps are reported, outliers are flagged in columns, and z-scored copies sit next to the
original counts.

**Prerequisite.** `medaesqa_v1.json` in the same folder. Download from https://osf.io/ydbzq/

## 1. Load and fingerprint

The hash is the whole point of freezing. If it changes, every number the team has
reported is suspect and we need to know immediately.

In [1]:
!pip install -q pandas

In [2]:
import json
import re
import hashlib
from pathlib import Path

import pandas as pd

RAW_PATH = Path("medaesqa_v1.json")
OUT_DIR = Path("processed")
OUT_DIR.mkdir(exist_ok=True)

SOURCE_SHA256 = hashlib.sha256(RAW_PATH.read_bytes()).hexdigest()

with open(RAW_PATH) as f:
    raw = json.load(f)

items = raw if isinstance(raw, list) else list(raw.values())

print("source sha256:", SOURCE_SHA256)
print("questions loaded:", len(items))

source sha256: a062742c1678b4379121ad030a06bda574afd376f331df41575938dd9849c383
questions loaded: 40


## 2. Field names

These are the real key names, confirmed against the file during EDA. They are written out
explicitly rather than guessed at, so that a rename upstream fails loudly instead of
silently producing empty columns.

In [3]:
# question_frame
K_FOCUS = "Focus"
K_TYPE = "Type"
K_TASK = "Task"
K_SUBJECT = "Subject matter (focus)"
K_BODY = "Body system"
K_SPECIALTY = "Specialty"

# machine answers
K_ACCURATE = "is_answer_accurate"
K_SENTENCES = "answer_sentences"
K_SENT_ID = "answer_sentence_id"
K_SENT_TEXT = "answer_sentence"
K_SENT_REL = "answer_sentence_relevance"
K_CITES = "citation_assessment"
K_PMID = "cited_pmid"
K_EXCERPT = "evidence_support"
K_RELATION = "evidence_relation"


def check_keys(sample, keys, label):
    """Fail loudly if the file no longer has the fields we expect."""
    missing = [k for k in keys if k not in sample]
    if missing:
        raise KeyError(f"{label}: expected keys not found: {missing}")
    print(f"{label}: all {len(keys)} expected keys present")


first = items[0]
check_keys(first["question_frame"], [K_FOCUS, K_TYPE, K_TASK, K_SUBJECT, K_BODY, K_SPECIALTY],
           "question_frame")

first_method = next(iter(first["machine_generated_answers"].values()))
check_keys(first_method, [K_ACCURATE, K_SENTENCES], "machine answer")
check_keys(first_method[K_SENTENCES][0], [K_SENT_ID, K_SENT_TEXT, K_SENT_REL, K_CITES],
           "answer sentence")

question_frame: all 6 expected keys present
machine answer: all 2 expected keys present
answer sentence: all 4 expected keys present


## 3. Questions

One cleaning decision here. `Specialty` and `Body system` hold multiple values in a single
string with inconsistent spacing, e.g. `pulmonology/ ENT / cardiology`. Counting those as
one category makes cardiology look rarer than it is.

We split on `/`, strip whitespace, and lowercase. The original strings are kept in
`*_raw` columns so nobody has to take our cleaning on faith.

One exception: `ob./gyn` is a single specialty that happens to contain a slash. Splitting
it naively produced two fake categories, `ob.` and `gyn`, so it is mapped to `ob-gyn`
before the split.

In [4]:
OBGYN_RE = re.compile(r"\bob\.?\s*/\s*gyn\b", re.IGNORECASE)


def split_multi(value):
    """'pulmonology/ ENT / cardiology' -> ['pulmonology', 'ent', 'cardiology']"""
    if not value:
        return []
    value = OBGYN_RE.sub("ob-gyn", str(value))
    return [p.strip().lower() for p in value.split("/") if p.strip()]


q_rows = []
for it in items:
    frame = it.get("question_frame", {}) or {}
    q_rows.append({
        "question_id": it.get("question_id"),
        "question": it.get("question", ""),
        "question_focus": frame.get(K_FOCUS),
        "question_type": frame.get(K_TYPE),
        "task": frame.get(K_TASK),
        "subject_matter": frame.get(K_SUBJECT),
        "body_system_raw": frame.get(K_BODY),
        "specialty_raw": frame.get(K_SPECIALTY),
        "body_systems": split_multi(frame.get(K_BODY)),
        "specialties": split_multi(frame.get(K_SPECIALTY)),
        "expert_answer": (it.get("expert_curated_answer") or "").strip(),
    })

questions = pd.DataFrame(q_rows)
questions["n_specialties"] = questions["specialties"].str.len()
questions["n_body_systems"] = questions["body_systems"].str.len()

# Lists don't survive a CSV round-trip cleanly, so store them pipe-joined.
questions["specialties"] = questions["specialties"].str.join("|")
questions["body_systems"] = questions["body_systems"].str.join("|")

print(f"{len(questions)} questions")
print(f"{(questions['n_specialties'] > 1).sum()} questions list more than one specialty")
questions.head(3)

40 questions
7 questions list more than one specialty


,question_id,question,question_focus,question_type,task,subject_matter,body_system_raw,specialty_raw,body_systems,specialties,expert_answer,n_specialties,n_body_systems
0,116,Are there ways to prevent sleep apnea or treat...,sleep apnea,Treatment,Decision support,clinical / problem,respiratory,pulmonology/ ENT / cardiology,respiratory,pulmonology|ent|cardiology,There are ways to prevent and treat sleep apne...,3,1
1,117,What will mutation in runx2 affect in the future?,runx2 mutation,Manifestation,Effect,genetics,full body,clinical genetics,full body,clinical genetics,The effect of the runx2 mutation depends on th...,1,1
2,118,how long does a minor cornea injury take to he...,corneal abrasion,Prognosis,Factoid,clinical / problem,visual,ophthalmology,visual,ophthalmology,Most corneal injuries heal in 24 to 72 hours [...,1,1


In [5]:
questions["task"] = questions["task"].str.strip().str.lower()
questions["question_type"] = questions["question_type"].str.strip().str.lower()
questions["subject_matter"] = questions["subject_matter"].str.strip().str.lower()

In [6]:
# Honest specialty counts, after splitting.
exploded = (questions["specialties"].str.split("|").explode().str.strip())
print(exploded.value_counts().head(15))

specialties
cardiology           9
clinical genetics    5
gp                   5
neurology            4
pulmonology          4
endocrinology        3
ophthalmology        2
ob-gyn               2
hepatology           2
psychiatry           2
ent                  1
gastroenterology     1
urology              1
rheumatology         1
sport medicine       1
Name: count, dtype: int64


## 4. Nuggets

Expert facts in `Predicate(subject, object)` form. These are gold for the Extractor.

We parse out the predicate and arguments where the pattern fits, and flag the ones that
don't. That flag matters: it tells the Extractor owner how much schema complexity is real
rather than assumed.

Two things the raw format hides:

- **Hedges.** Many nuggets open with a bracketed modality, e.g. `[may] Prevent (...)`. It is
  pulled into a `modality` column so the nugget still counts as fitting the pattern.
- **CUIs are inline.** They are written inside the arguments, e.g. `sleep apnea [C0037315]`,
  not stored in a separate field, so they are extracted from the text.

In [7]:
NUGGET_RE = re.compile(r"^\s*(?:\[([^\]]+)\]\s*)?([A-Za-z_ ]+?)\s*\((.*)\)\s*$")
CUI_RE = re.compile(r"\bC\d{7}\b")

nug_rows = []
for it in items:
    qid = it.get("question_id")
    for idx, n in enumerate(it.get("nuggets") or []):
        if isinstance(n, str):
            body, cuis = n, []
        else:
            body = n.get("nugget") or n.get("text") or ""
            cuis = n.get("cuis") or n.get("CUI") or []
            if isinstance(cuis, str):
                cuis = [cuis]
        cuis = list(dict.fromkeys([*map(str, cuis), *CUI_RE.findall(body)]))

        m = NUGGET_RE.match(body)
        nug_rows.append({
            "question_id": qid,
            "nugget_idx": idx,
            "nugget": body,
            "modality": m.group(1).strip().lower() if m and m.group(1) else None,
            "predicate": m.group(2).strip() if m else None,
            "arguments": m.group(3).strip() if m else None,
            "n_arguments": len(m.group(3).split(",")) if m else None,
            "fits_predicate_pattern": bool(m),
            "cuis": "|".join(str(c) for c in cuis),
            "n_cuis": len(cuis),
        })

nuggets = pd.DataFrame(nug_rows)
print(f"{len(nuggets)} nuggets across {nuggets['question_id'].nunique()} questions")
print(f"{(~nuggets['fits_predicate_pattern']).mean() * 100:.1f}% don't fit Predicate(a, b)")
print(f"{(nuggets['n_cuis'] > 0).mean() * 100:.1f}% carry at least one CUI")
nuggets.head(3)

498 nuggets across 40 questions
0.6% don't fit Predicate(a, b)
100.0% carry at least one CUI


,question_id,nugget_idx,nugget,modality,predicate,arguments,n_arguments,fits_predicate_pattern,cuis,n_cuis
0,116,0,"Exists (treat naturally[C0027495], sleep apnea...",None,Exists,"treat naturally[C0027495], sleep apnea [C0037315]",2.0,True,C0027495|C0037315,2
1,116,1,"[may] Prevent (lifestyle changes [C0870811], s...",may,Prevent,"lifestyle changes [C0870811], sleep apnea [C00...",2.0,True,C0870811|C0037315,2
2,116,2,"[may] Treat (lifestyle changes [C0870811], sle...",may,Treat,"lifestyle changes [C0870811], sleep apnea [C00...",2.0,True,C0870811|C0037315,2


## 5. Sentences

One row per answer sentence. This is the table the Verifier calibrates against.

Two decisions recorded in columns rather than applied destructively:

- **Unlabeled sentences stay in.** 244 sentences carry no relevance grade, and they all
  come from answers already rejected as inaccurate. We flag them with `has_relevance_label`
  instead of dropping them, so whoever computes kappa can exclude them and see why.
- **Nothing is imputed.** A missing label means a human did not grade it. Filling that in
  would be inventing data.

In [8]:
def norm(x):
    return str(x).strip().lower() if x is not None else None


sent_rows = []
for it in items:
    qid = it.get("question_id")
    for method_id, ans in (it.get("machine_generated_answers") or {}).items():
        accurate = norm(ans.get(K_ACCURATE))
        for idx, s in enumerate(ans.get(K_SENTENCES) or []):
            sent_rows.append({
                "question_id": qid,
                "method_id": method_id,
                "sentence_idx": idx,
                "sentence_id": s.get(K_SENT_ID),
                "sentence": (s.get(K_SENT_TEXT) or "").strip(),
                "relevance": norm(s.get(K_SENT_REL)),
                "answer_is_accurate": accurate,
                "n_citations": len(s.get(K_CITES) or []),
            })

sentences = pd.DataFrame(sent_rows)
sentences["has_relevance_label"] = sentences["relevance"].notna()

print(f"{len(sentences)} sentences")
print(f"{sentences['has_relevance_label'].sum()} labeled -> the Verifier calibration pool")
print()
print(sentences["relevance"].value_counts(dropna=False))

5162 sentences
4918 labeled -> the Verifier calibration pool

relevance
required         3958
unnecessary       698
None              244
borderline        243
inappropriate      19
Name: count, dtype: int64


In [9]:
# Confirm the unlabeled sentences really are all from rejected answers.
crosstab = pd.crosstab(sentences["answer_is_accurate"].fillna("(none)"),
                       sentences["has_relevance_label"],
                       rownames=["answer_is_accurate"], colnames=["has_label"])
print(crosstab)

has_label           False  True 
answer_is_accurate              
no                    244      0
yes                     0   4918


## 6. Citations

This is the one place the raw file is genuinely ambiguous, so the rule below is a
decision the team should be able to see and argue with.

A single sentence-PMID pair can appear several times, once per supporting excerpt, and in
390 cases those excerpts carry **different verdicts**. Keeping all rows inflates the
count to 9,611 against the paper's 8,111. Deduplicating arbitrarily picks a verdict at
random.

**Our rule:** collapse to one row per sentence-PMID pair, and when verdicts conflict, the
most severe one wins. Contradiction beats support, because the paper's own definition is
that a document supports a claim only when nothing in it contradicts the claim.

This yields 8,124 citations at 66.8% supporting, against the paper's 8,111 at 67.7%.
The residual gap is unexplained and is on our question list for the advisor, most likely
related to the `invalid citation` label, which appears in the file but not in the paper.

In [11]:
# Most severe first. Anything unrecognized sorts last and is left untouched.
RELATION_PRIORITY = ["contradicts", "supports", "neutral", "not_relevant", "invalid citation"]
RELATION_RANK = {lab: i for i, lab in enumerate(RELATION_PRIORITY)}

# The file uses present participles where the paper uses base forms.
RELATION_MAP = {
    "supporting": "supports",
    "supports": "supports",
    "contradicting": "contradicts",
    "contradicts": "contradicts",
    "neutral": "neutral",
    "not relevant": "not_relevant",
    "not_relevant": "not_relevant",
    "invalid citation": "invalid citation",
}


def norm_relation(x):
    lab = norm(x)
    return RELATION_MAP.get(lab, lab)


cit_rows = []
for it in items:
    qid = it.get("question_id")
    for method_id, ans in (it.get("machine_generated_answers") or {}).items():
        for idx, s in enumerate(ans.get(K_SENTENCES) or []):
            for c in (s.get(K_CITES) or []):
                cit_rows.append({
                    "question_id": qid,
                    "method_id": method_id,
                    "sentence_idx": idx,
                    "pmid": c.get(K_PMID),
                    "evidence_relation": norm_relation(c.get(K_RELATION)),
                    "evidence_excerpt": (c.get(K_EXCERPT) or "").strip(),
                })

citations_raw = pd.DataFrame(cit_rows)
print(f"{len(citations_raw)} raw citation_assessment rows")

PAIR_KEYS = ["question_id", "method_id", "sentence_idx", "pmid"]

conflicts = citations_raw.groupby(PAIR_KEYS)["evidence_relation"].nunique()
n_conflicts = int((conflicts > 1).sum())
print(f"{n_conflicts} pairs carry conflicting verdicts across excerpts")

citations = (citations_raw
             .assign(_rank=citations_raw["evidence_relation"].map(RELATION_RANK).fillna(99))
             .sort_values("_rank")
             .drop_duplicates(PAIR_KEYS)
             .drop(columns="_rank")
             .sort_values(PAIR_KEYS)
             .reset_index(drop=True))

# Keep the excerpt count, since it's information the collapse would otherwise throw away.
excerpt_counts = citations_raw.groupby(PAIR_KEYS).size().rename("n_excerpts")
citations = citations.merge(excerpt_counts, on=PAIR_KEYS, how="left")

print(f"{len(citations)} unique sentence-PMID pairs")
print()
print(citations["evidence_relation"].value_counts(normalize=True).round(3))

9611 raw citation_assessment rows
390 pairs carry conflicting verdicts across excerpts
8124 unique sentence-PMID pairs

evidence_relation
supports            0.668
not_relevant        0.191
neutral             0.069
invalid citation    0.050
contradicts         0.022
Name: proportion, dtype: float64


## 7. Missing values

Every column of every output table is checked for nulls and blank strings, and each gap is
tied to a reason. **Nothing is imputed.** In this dataset a missing value always means
either "a human did not grade this" or "this field does not apply" (a nugget with no hedge,
a citation with no excerpt). Filling those in would invent labels the Verifier is later
calibrated against.

If the `reason` column ever says `UNEXPECTED`, a gap has appeared that we have not
explained. Look at it before writing outputs.

In [12]:
# Why each known gap exists. Anything not listed here is unexpected and worth a look.
EXPECTED_GAPS = {
    ("sentences", "relevance"): "never graded: sentence is from an answer rejected as inaccurate",
    ("nuggets", "modality"): "nugget has no hedge like [may]",
    ("nuggets", "predicate"): "nugget does not fit Predicate(a, b)",
    ("nuggets", "arguments"): "nugget does not fit Predicate(a, b)",
    ("nuggets", "n_arguments"): "nugget does not fit Predicate(a, b)",
    ("nuggets", "cuis"): "nugget has no inline CUI tag",
    ("citations", "evidence_excerpt"): "annotator wrote no excerpt for this citation",
}

tables = {"questions": questions, "nuggets": nuggets,
          "sentences": sentences, "citations": citations}

rows = []
for table, df in tables.items():
    for col in df.columns:
        s = df[col]
        # Blank strings are how missing text survives the flatten, so they count too.
        n_missing = int((s.isna() | s.astype(str).str.strip().eq("")).sum())
        if n_missing:
            rows.append({
                "table": table,
                "column": col,
                "n_missing": n_missing,
                "pct_missing": round(n_missing / len(df) * 100, 1),
                "reason": EXPECTED_GAPS.get((table, col), "UNEXPECTED - investigate"),
            })

missing = pd.DataFrame(rows, columns=["table", "column", "n_missing", "pct_missing", "reason"])
print(f"{len(tables)} tables, {sum(len(df.columns) for df in tables.values())} columns checked, "
      f"{len(missing)} with gaps")
missing

4 tables, 39 columns checked, 7 with gaps


,table,column,n_missing,pct_missing,reason
0,questions,question_focus,1,2.5,UNEXPECTED - investigate
1,nuggets,modality,329,66.1,nugget has no hedge like [may]
2,nuggets,predicate,3,0.6,"nugget does not fit Predicate(a, b)"
3,nuggets,arguments,3,0.6,"nugget does not fit Predicate(a, b)"
4,nuggets,n_arguments,3,0.6,"nugget does not fit Predicate(a, b)"
5,sentences,relevance,244,4.7,never graded: sentence is from an answer rejec...
6,citations,evidence_excerpt,1959,24.1,annotator wrote no excerpt for this citation


## 8. Outliers

The only numeric values here are lengths we derive from the text: words per expert answer,
words per machine-answer sentence, and words per evidence excerpt. We flag anything outside
1.5 x IQR in a `*_outlier` column.

**Nothing is removed.** This is a frozen evaluation set: dropping rows would break every
comparison against the paper, and a long sentence is still a real answer the Verifier has
to judge. The flags exist so downstream owners can check whether their metrics behave
differently on the extremes, for example whether readability scores blow up on very long
sentences.

Count columns like `n_citations` and `n_specialties` are not flagged: almost every value is
1, so the IQR is zero and the rule would mark every other value as an outlier.

In [13]:
def iqr_outliers(s, k=1.5):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (s < q1 - k * iqr) | (s > q3 + k * iqr)


questions["expert_answer_n_words"] = questions["expert_answer"].str.split().str.len()
sentences["sentence_n_words"] = sentences["sentence"].str.split().str.len()
# A blank excerpt is a missing value, not a zero-length one, so it must not drag the quartiles down.
citations["excerpt_n_words"] = (citations["evidence_excerpt"].str.split().str.len()
                                 .where(citations["evidence_excerpt"].str.len() > 0))

OUTLIER_COLS = [
    ("questions", questions, "expert_answer_n_words"),
    ("sentences", sentences, "sentence_n_words"),
    ("citations", citations, "excerpt_n_words"),
]

summary = []
for table, df, col in OUTLIER_COLS:
    df[f"{col}_outlier"] = iqr_outliers(df[col])
    s = df[col].dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    summary.append({
        "table": table,
        "column": col,
        "min": s.min(),
        "median": s.median(),
        "max": s.max(),
        "lower_fence": round(q1 - 1.5 * (q3 - q1), 1),
        "upper_fence": round(q3 + 1.5 * (q3 - q1), 1),
        "n_outliers": int(df[f"{col}_outlier"].sum()),
        "pct_outliers": round(df[f"{col}_outlier"].mean() * 100, 1),
    })

pd.DataFrame(summary)

,table,column,min,median,max,lower_fence,upper_fence,n_outliers,pct_outliers
0,questions,expert_answer_n_words,45.0,88.0,204.0,13.5,153.5,3,7.5
1,sentences,sentence_n_words,1.0,21.0,61.0,3.5,39.5,124,2.4
2,citations,excerpt_n_words,5.0,26.0,283.0,-6.5,61.5,249,3.1


In [14]:
# Eyeball the extremes before trusting the flags: are they data errors or just long answers?
cols = ["question_id", "method_id", "sentence_n_words", "relevance", "sentence"]
print("Longest sentences:")
print(sentences.nlargest(5, "sentence_n_words")[cols].to_string(index=False))
print()
print("Shortest sentences:")
print(sentences.nsmallest(5, "sentence_n_words")[cols].to_string(index=False))

Longest sentences:
question_id method_id  sentence_n_words   relevance                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 sentence
        145       M18                61    required Other side effects may include gastrointestinal-associated effects (constipation, hypersalivation, oropharyngeal lesions, nasal congestion, nausea, nocturnal enuresis, and urinary retention), metabolic effects (obesity, insulin resistance, dyslipidemia, impaired glucose tolerance, and hypertension), neuromuscular effects (

## 9. Normalization

MedAESQA has no measured numeric features, only labels and text. The only numbers are counts
we derived (words per sentence, citations per sentence, and so on), and nothing in our
pipeline trains on them, so there is nothing that strictly needs scaling.

For anyone who wants to compare or plot them on one scale, we add z-scored copies,
`(x - mean) / std`, in `*_z` columns. The original counts are left untouched, because the
raw numbers are what match the paper. Missing values stay missing.

In [15]:
def zscore(s):
    std = s.std()
    return ((s - s.mean()) / std).round(3) if std else s * 0.0


Z_COLS = [
    (questions, "expert_answer_n_words"),
    (sentences, "sentence_n_words"),
    (sentences, "n_citations"),
    (citations, "excerpt_n_words"),
]

for df, col in Z_COLS:
    df[f"{col}_z"] = zscore(df[col])

pd.DataFrame([
    {"column": f"{col}_z", "mean": round(df[f"{col}_z"].mean(), 3),
     "std": round(df[f"{col}_z"].std(), 3)}
    for df, col in Z_COLS
])

,column,mean,std
0,expert_answer_n_words_z,-0.0,1.0
1,sentence_n_words_z,0.0,1.0
2,n_citations_z,-0.0,1.0
3,excerpt_n_words_z,-0.0,1.0


## 10. Validation

Every number below is checked against the published paper. If any line says FAIL, do not
commit the output, because something upstream has changed.

In [16]:
EXPECTED = {
    "questions": (len(questions), 40),
    "sentences": (len(sentences), 5162),
    "required sentences": (int((sentences["relevance"] == "required").sum()), 3958),
    "evidence excerpts": (int((citations_raw["evidence_excerpt"].str.len() > 0).sum()), 7651),
}

all_ok = True
for name, (got, want) in EXPECTED.items():
    ok = got == want
    all_ok = all_ok and ok
    print(f"{name:22s} got {got:6d}  paper {want:6d}   {'OK' if ok else 'FAIL'}")

print()
print(f"citations             got {len(citations):6d}  paper   8111   "
      f"(known {len(citations) - 8111:+d} gap, see section 6)")
print()
print("ALL EXACT CHECKS PASSED" if all_ok else "SOMETHING CHANGED - investigate before committing")

questions              got     40  paper     40   OK
sentences              got   5162  paper   5162   OK
required sentences     got   3958  paper   3958   OK
evidence excerpts      got   7651  paper   7651   OK

citations             got   8124  paper   8111   (known +13 gap, see section 6)

ALL EXACT CHECKS PASSED


## 11. Write outputs

In [17]:
outputs = {
    "medaesqa_questions.csv": questions,
    "medaesqa_nuggets.csv": nuggets,
    "medaesqa_sentences.csv": sentences,
    "medaesqa_citations.csv": citations,
}

for name, df in outputs.items():
    path = OUT_DIR / name
    df.to_csv(path, index=False)
    print(f"{name:30s} {len(df):6d} rows  {len(df.columns):2d} cols")

manifest = [
    "MedAESQA processed data - FaithfulMed (Google 2D)",
    "Prepared by: Isha, Isabella",
    "",
    f"source file:   medaesqa_v1.json",
    f"source sha256: {SOURCE_SHA256}",
    "",
    "Row counts:",
]
for name, df in outputs.items():
    manifest.append(f"  {name:32s} {len(df)}")
manifest += [
    "",
    "Cleaning rules applied:",
    "  - Specialty and Body system split on '/', stripped, lowercased.",
    "    'ob./gyn' mapped to 'ob-gyn' before splitting. Originals kept in *_raw columns.",
    "  - Nugget hedges like '[may]' moved to a modality column; CUIs extracted",
    "    from inline [Cxxxxxxx] tags.",
    "  - Sentences with no relevance label retained and flagged via",
    f"    has_relevance_label. All {int((~sentences['has_relevance_label']).sum())} "
    "come from answers marked not accurate.",
    "  - Citations collapsed to one row per sentence-PMID pair. On conflicting",
    "    verdicts the most severe wins: contradicts > supports > neutral >",
    f"    not_relevant > invalid citation. {n_conflicts} pairs were affected.",
    "  - Missing values audited, never imputed (see notebook section 7).",
    "  - Outliers flagged in *_outlier columns (1.5 x IQR rule), never removed.",
    "  - Numeric counts z-scored into *_z columns; originals left untouched.",
    "",
    "Known open question: our citation count is "
    f"{len(citations)} vs the paper's 8111. Likely related to the",
    "'invalid citation' label, which appears in the file but not the paper. Raised with advisor.",
    "",
    "Citation: Gupta, D., Bartels, D. & Demner-Fushman, D. Scientific Data 12, 1035 (2025). CC BY 4.0.",
]

manifest_path = OUT_DIR / "medaesqa_freeze.txt"
manifest_path.write_text("\n".join(manifest))
print()
print(manifest_path.read_text())

medaesqa_questions.csv             40 rows  16 cols
medaesqa_nuggets.csv              498 rows  10 cols
medaesqa_sentences.csv           5162 rows  13 cols
medaesqa_citations.csv           8124 rows  10 cols

MedAESQA processed data - FaithfulMed (Google 2D)
Prepared by: Isha, Isabella

source file:   medaesqa_v1.json
source sha256: a062742c1678b4379121ad030a06bda574afd376f331df41575938dd9849c383

Row counts:
  medaesqa_questions.csv           40
  medaesqa_nuggets.csv             498
  medaesqa_sentences.csv           5162
  medaesqa_citations.csv           8124

Cleaning rules applied:
  - Specialty and Body system split on '/', stripped, lowercased.
    'ob./gyn' mapped to 'ob-gyn' before splitting. Originals kept in *_raw columns.
  - Nugget hedges like '[may]' moved to a modality column; CUIs extracted
    from inline [Cxxxxxxx] tags.
  - Sentences with no relevance label retained and flagged via
    has_relevance_label. All 244 come from answers marked not accurate.
  - Citations

## 12. Data dictionary



**medaesqa_questions.csv** — one row per question (40)
`question_id`, `question`, `question_focus`, `question_type`, `task`, `subject_matter`,
`body_system_raw`, `specialty_raw`, `body_systems` and `specialties` (pipe-joined, cleaned),
`n_specialties`, `n_body_systems`, `expert_answer`, `expert_answer_n_words`,
`expert_answer_n_words_outlier`, `expert_answer_n_words_z`

**medaesqa_nuggets.csv** — one row per expert fact (498)
`question_id`, `nugget_idx`, `nugget`, `modality` (e.g. `may`, or null), `predicate`,
`arguments`, `n_arguments`, `fits_predicate_pattern`, `cuis` (pipe-joined), `n_cuis`

**medaesqa_sentences.csv** — one row per answer sentence (5,162)
`question_id`, `method_id` (M1-M30), `sentence_idx`, `sentence_id`, `sentence`,
`relevance` (required / unnecessary / borderline / inappropriate / null),
`answer_is_accurate`, `n_citations`, `has_relevance_label`, `sentence_n_words`,
`sentence_n_words_outlier`, `sentence_n_words_z`, `n_citations_z`

**medaesqa_citations.csv** — one row per sentence-PMID pair (~8,124)
`question_id`, `method_id`, `sentence_idx`, `pmid`, `evidence_relation`
(supports / contradicts / neutral / not_relevant / invalid citation), `evidence_excerpt`,
`n_excerpts`, `excerpt_n_words`, `excerpt_n_words_outlier`, `excerpt_n_words_z`

`*_outlier` columns are flags only (1.5 x IQR rule); no rows were removed. `*_z` columns are
z-scores within each table, `(x - mean) / std`.

Join sentences to citations on `question_id` + `method_id` + `sentence_idx`.